# Volatility Regression 03 — nested-block next-5m volatility OLS

**Question.** Which groups of current-bar information add out-of-sample linear forecast value for the volatility of the next five-minute bar, beyond the bar's own volatility?

**Design** (VR 007, approved by Ivan on 2026-10-09):

- **Targets:** paired, as in 01. `parkinson` is $\log(H/L)/\sqrt{4\log 2}$ from the next perpetual trade bar; `index_abs` is $|r^{\text{index}}_{t+1}|/\sqrt{2/\pi}$. The response is $Y=\log(\sigma+\epsilon_{\text{train}})$.
- **No development cutoff:** this branch is independent of HSMM.
- **Excluded by design:** option smile shape, fitted smile, wings and spreads, and `atm_iv_d60`. On a 0DTE chain these do not exist in the hours before expiry. L2 instability statistics are also excluded, because they are absent before 2026-10-02.
- **Nested blocks:**
  - B0: returns, volatility and clock;
  - B1: B0 plus flow and basis/OI;
  - B2: B1 plus the L2 time-weighted book;
  - B3: B2 plus ATM IV level and short ATM IV changes.
- **Coverage and budget rules:**
  - A feature is admitted per asset if its coverage on development rows is at least 80%.
  - A fit is attempted only if $n_{\text{train}} \ge 20p$.
  - `log_rv_4h_tol` (at least 44 of 48 index returns, rescaled) replaces the shared strict 48/48 column.
- **Evaluation:**
  - The last 25% of sessions are the test set and are scored once.
  - Earlier sessions are scored by forward chaining: train on sessions $1..k$, validate on session $k+1$.
  - Every block is scored on its **own** complete rows and on rows **common** to all blocks; the common rows are where block increments are comparable.
- **References:** train mean (the benchmark for $R^2_{\text{OOS}}$) and persistence. There is no HAR study.
- **Deletion:** greedy backward deletion inside B2. A removal is accepted only if it lowers the mean fold RMSE **and** helps in a majority of folds. The test set is never used.
- **Collector vintage:** test residuals are reported by data-derived collector vintage, read from `02_vintage_boundaries.csv`. **Run notebook 02 first.**

## 1. Environment and data (DuckDB read-only)

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "src" / "data_engineering").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Launch from the repository root or one of its subfolders.")
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
DATA_ROOT = os.getenv("DERIBIT_DATA_ROOT")
if not DATA_ROOT:
    raise RuntimeError("DERIBIT_DATA_ROOT must be set in .env.")
DB_PATH = Path(DATA_ROOT) / "database" / "research.duckdb"
if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)

from data_engineering.modeling import vol_ols_5m_blocks as vb
from data_engineering.modeling import vol_ols_5m as base
OUTPUT = ROOT / "notebooks" / "Volatility_Regression"

TEST_FRACTION = 0.25
MIN_TRAIN_SESSIONS = 6
DELETE_BLOCK = "B2_l2_book"
MAX_DROPS = 15

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    con.execute("SET TimeZone = 'UTC'")
    df = con.execute("""
        SELECT * FROM time_series_feature_state
        ORDER BY underlying, state_time
    """).df()
ASSETS = tuple(sorted(df["underlying"].dropna().astype(str).unique()))
vintage_path = OUTPUT / "02_vintage_boundaries.csv"
boundaries = pd.read_csv(vintage_path) if vintage_path.is_file() else None
if boundaries is None:
    print("WARNING: 02_vintage_boundaries.csv not found; vintages will be 'unknown'. Run 02 first.")
print("Blocks version:", vb.VOL_OLS_BLOCKS_VERSION, "| base:", base.VOL_OLS_5M_VERSION)
print("Assets:", ASSETS, "| rows:", len(df))
print("Span:", df.state_time.min(), "to", df.state_time.max())

In [ ]:
for pattern in ("test_vol_ols_5m.py", "test_vol_ols_5m_blocks.py"):
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(ROOT / "tests"),
                    "-p", pattern, "-v"], cwd=ROOT, check=True)

## 2. Fit blocks per asset and target

In [ ]:
fold_frames, admission_frames, clock_frames, loss_frames = [], [], [], []
deletion_frames, reduced_rows, vintage_rows, plan_rows, errors = [], [], [], [], []

for asset in ASSETS:
    g = df[df["underlying"] == asset]
    try:
        meta, X = vb.panel(g)
        valid = vb.paired_valid(meta)
        plan = vb.session_plan(meta, valid, TEST_FRACTION, MIN_TRAIN_SESSIONS)
        lab = plan["labels"]
        dev_rows = np.isin(lab, plan["dev"]) & valid
        test_rows = np.isin(lab, plan["test"]) & valid
        available, adm = vb.admit_features(X, dev_rows)
        adm["underlying"] = asset
        admission_frames.append(adm)
        vint = vb.assign_vintage(meta["state_time"], boundaries, asset)
        plan_rows.append({"underlying": asset, "sessions": len(plan["sessions"]),
                          "dev_first": plan["dev"][0], "dev_last": plan["dev"][-1],
                          "test_first": plan["test"][0], "test_last": plan["test"][-1],
                          "folds": len(plan["folds"]), "paired_valid_rows": int(valid.sum()),
                          "test_vintages": ",".join(sorted(set(vint[test_rows])))})
        cc = vb.clock_coverage(meta, X, valid, available)
        cc["underlying"] = asset
        clock_frames.append(cc)
        ul = vb.unique_loss(X, dev_rows, vb.nested_features(vb.BLOCK_ORDER[-1], available))
        ul["underlying"] = asset
        loss_frames.append(ul)
    except Exception as exc:
        errors.append({"underlying": asset, "target": "*", "reason": f"{type(exc).__name__}: {exc}"})
        print(f"{asset:>6} NOT PLANNED: {errors[-1]['reason']}")
        continue

    for target in base.TARGETS:
        try:
            res, final = vb.evaluate_blocks(meta, X, valid, plan, target, available)
            res.insert(0, "underlying", asset)
            fold_frames.append(res)

            for (population, block), r in final.items():
                resid = r["y"][r["idx"]] - r["pred"]
                for v in sorted(set(vint[r["idx"]])):
                    m = vint[r["idx"]] == v
                    vintage_rows.append({"underlying": asset, "target": target,
                                         "population": population, "model": block,
                                         "vintage": v, "n": int(m.sum()),
                                         "mean_residual": float(resid[m].mean()),
                                         "rmse": float(np.sqrt(np.mean(resid[m] ** 2)))})

            start = vb.nested_features(DELETE_BLOCK, available)
            keep, hist = vb.backward_by_folds(meta, X, valid, plan, target, start, MAX_DROPS)
            if len(hist):
                hist.insert(0, "underlying", asset)
                deletion_frames.append(hist)
            complete = np.isfinite(X[start].to_numpy(float)).all(axis=1)
            y, pers, _ = vb.target_y(meta, target, dev_rows)
            for label, names in ((DELETE_BLOCK + "_start", start), (DELETE_BLOCK + "_reduced", keep)):
                r = vb.fit_score(X, y, dev_rows & complete, test_rows & complete, names)
                rec = {"underlying": asset, "target": target, "model": label,
                       "p": len(r["names"]), "n_train": r["n_train"], "n_test": r["n_score"],
                       "fitted": r["ok"], "reason": r.get("reason", "")}
                if r["ok"]:
                    ps = base.score(y[r["idx"]], pers[r["idx"]], r["train_mean"], "test")
                    rec.update({"log_rmse": r["log_rmse"], "r2_vs_train_mean": r["r2_vs_train_mean"],
                                "persistence_log_rmse": ps["log_rmse"],
                                "rmse_ratio_vs_persistence": r["log_rmse"] / ps["log_rmse"]})
                reduced_rows.append(rec)
            print(f"{asset:>6} {target:<10} folds={len(plan['folds'])} "
                  f"B2 p={len(start)} -> {len(keep)}")
        except Exception as exc:
            errors.append({"underlying": asset, "target": target,
                           "reason": f"{type(exc).__name__}: {exc}"})
            print(f"{asset:>6} {target:<10} NOT FIT: {errors[-1]['reason']}")

folds_df = pd.concat(fold_frames, ignore_index=True) if fold_frames else pd.DataFrame()
admission_df = pd.concat(admission_frames, ignore_index=True) if admission_frames else pd.DataFrame()
clock_df = pd.concat(clock_frames, ignore_index=True) if clock_frames else pd.DataFrame()
loss_df = pd.concat(loss_frames, ignore_index=True) if loss_frames else pd.DataFrame()
deletion_df = pd.concat(deletion_frames, ignore_index=True) if deletion_frames else pd.DataFrame()
reduced_df = pd.DataFrame(reduced_rows)
vintage_df = pd.DataFrame(vintage_rows)
plan_df = pd.DataFrame(plan_rows)
errors_df = pd.DataFrame(errors)
display(plan_df)
if not errors_df.empty:
    display(errors_df)

## 3. Development folds: block increments on common rows

RMSE ratio below 1 means the block beats persistence on the same rows. The fold-to-fold spread is the uncertainty that matters with this few sessions.

In [ ]:
if not folds_df.empty:
    f = folds_df[(folds_df.stage == "fold") & folds_df.fitted]
    summ = (f.groupby(["underlying", "target", "population", "model"])["rmse_ratio_vs_persistence"]
              .agg(["mean", "std", "count"]).round(3))
    display(summ.xs("common", level="population") if "common" in f.population.values else summ)
    print("Admitted features per asset (development coverage >= 80%):")
    display(admission_df.pivot_table(index="feature", columns="underlying",
                                     values="admitted", aggfunc="first"))

## 4. Final test (scored once)

In [ ]:
if not folds_df.empty:
    t = folds_df[(folds_df.stage == "test") & folds_df.fitted]
    display(t.pivot_table(index=["underlying", "target", "population"], columns="model",
                          values=["r2_vs_train_mean", "rmse_ratio_vs_persistence"]).round(3))
    display(t.pivot_table(index=["underlying", "target", "population"], columns="model",
                          values="n_score"))
display(reduced_df.round(3))

## 5. Deletion path, coverage by hour, marginal bottlenecks, residuals by collector vintage

In [ ]:
if not deletion_df.empty:
    display(deletion_df)
if not clock_df.empty:
    display(clock_df.pivot_table(index="utc_hour", columns=["model"], values="share",
                                 aggfunc="median").round(2))
if not loss_df.empty:
    display(loss_df.groupby("feature")["unique_loss_rows"].median().sort_values(ascending=False).head(10))
if not vintage_df.empty:
    display(vintage_df[vintage_df.population == "own"].pivot_table(
        index=["underlying", "target", "vintage"], columns="model", values="mean_residual").round(3))

## 6. Save outputs (no database writes)

In [ ]:
outputs = {
    "03_plan.csv": plan_df,
    "03_feature_admission.csv": admission_df,
    "03_block_scores.csv": folds_df,
    "03_reduced_test.csv": reduced_df,
    "03_deletion_history.csv": deletion_df,
    "03_clock_coverage.csv": clock_df,
    "03_unique_loss.csv": loss_df,
    "03_test_residual_by_vintage.csv": vintage_df,
    "03_fit_errors.csv": errors_df,
}
for name, table in outputs.items():
    table.to_csv(OUTPUT / name, index=False)
    print(name, "rows:", len(table))
summary = {
    "version": vb.VOL_OLS_BLOCKS_VERSION,
    "base_version": base.VOL_OLS_5M_VERSION,
    "targets": list(base.TARGETS),
    "blocks": {b: vb.BLOCKS[b] for b in vb.BLOCK_ORDER},
    "excluded_by_design": list(vb.EXCLUDED_BY_DESIGN),
    "feature_min_coverage_dev": vb.FEATURE_MIN_COVERAGE,
    "min_rows_per_param": vb.MIN_ROWS_PER_PARAM,
    "test_fraction": TEST_FRACTION, "min_train_sessions": MIN_TRAIN_SESSIONS,
    "delete_block": DELETE_BLOCK, "max_drops": MAX_DROPS,
    "data_span": [str(df.state_time.min()), str(df.state_time.max())],
    "vintage_boundaries_file": vintage_path.name if boundaries is not None else None,
    "failed": errors,
}
(OUTPUT / "03_run_metadata.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Done. Commit the notebook and 03_* outputs.")

## Interpretation contract

1. A block's value is its increment over the previous block on **common** rows, read across folds, not one test number.
2. A feature that is not admitted, or a block that fails the parameter budget, has no evidence either way.
3. A block that helps under `parkinson` but not under `index_abs` (e.g. flow) may be predicting the trade-range measurement rather than volatility.
4. If B2 residuals differ systematically by collector vintage, L2 coefficients are not stable across the event-time fix and must not be interpreted.
5. Research diagnostics only.